# normalisation.ipynb — Reconstructing `flood_composite_6h_ahead`

**Context.** The original code that merged `soil_moisture` (ERA5 `swvl1`) and
`precipitation` (IMERG) into the `flood_composite` target — and then
normalised it — was lost. This notebook documents the reverse-engineering
process used to reconstruct an *approximate* version of that target directly
from `merged_output_hourly_imerg/merged_data_normalised.csv`, and builds the
data needed to train experimental models (`ConvLSTM_approx`, `CNN3D_approx`)
on it.

**This is an approximation, not a recovery of the original formula.**
Validated out-of-sample R² is **0.74** (Random Forest, spatial-neighbourhood
features) — a substantial improvement over an initial local-only linear fit
(R²=0.31 out-of-sample), but still leaves ~26% of variance unexplained.
Every artefact produced here (target arrays, checkpoints, plots) lives in
files/directories suffixed `_approx` or under `experimental_approx_target/`
so the original `ML_MODELS.ipynb` results, checkpoints, and CSV are never
touched or overwritten.

## 1. Why the target needed reconstructing

Plotting `test_quantile_fan.png` showed the "actual" curve pinned near a flat
~40 for long stretches before jumping abruptly to ~68+. Histogramming the
entire test set's `flood_composite_6h_ahead` values (21.9M valid cells)
showed why: there is a **hard gap in the distribution between ~40 and ~52**
— essentially zero recorded values in that range anywhere in the dataset.
That is a property of the target itself, not a plotting bug.

In [ ]:
import numpy as np

SEAGATE = r"W:\Dissertation\Volumes\Seagate_2TB"
test_y = np.load(SEAGATE + r"\test_y.npy")

valid = ~np.isnan(test_y)
vals = test_y[valid]
hist, edges = np.histogram(vals, bins=50)
for c, e in zip(hist, edges):
    print(f"{e:8.2f}: {c}")

## 2. Identifying the two source columns

The user recalled merging `soil_moisture_1` and `precipitation` into the
composite. The CSV header's closest matches are:

- `era5_swvl1` — ERA5's standard short name for volumetric soil water
  layer 1 (soil moisture)
- `imerg_precipitation` — IMERG rainfall

There is no standalone `flood_composite` column (only the already-shifted
`flood_composite_6h_ahead`), and no script anywhere in the repo
(`merged_datas.ipynb`, `IMERG.ipynb`, `ERA5-1.ipynb`, `Synthetic_data_gen.ipynb`)
computes it — it arrives pre-existing before any notebook here touches it.

## 3. Reconstructing the un-shifted base value

Since `flood_composite_6h_ahead` at row `h` equals the base composite at row
`h+6` for the same grid cell, the base value at hour `h` can be recovered as
`target(h-6)`. This lets us align the base composite with `era5_swvl1` and
`imerg_precipitation` at the *same* row `h` without needing the original
(lost) pre-shift column.

## 4. Hypothesis search

Streaming increasing amounts of the CSV and regressing the reconstructed
base composite against candidate features:

| Hypothesis | R² |
|---|---|
| `swvl1(h)` + `precip(h)` instantaneous | 0.13 |
| `swvl1(h)` + precip summed over trailing 6h | 0.20 |
| `swvl1(h)` + precip summed over trailing 24h | 0.49 |
| `swvl1(h)` + `log1p(precip summed over trailing 24h)` | 0.63 (same-window fit) |

A window sweep (6h → 48h) peaked cleanly at **24 hours** — consistent with a
real antecedent-rainfall / soil-saturation flood-risk driver.

**Critical check — out-of-sample validation.** Fitting on hours [24, 450) and
testing on unseen, later hours [450, 594) dropped this linear formula's R²
from 0.57 (train) to **0.31 (held-out)** — the initial 0.63 figure was
measured on the same window it was fit on and did not generalise.

**Improvement — spatial neighbourhood features.** Flood risk is a
catchment-scale phenomenon, not a single-pixel one. Adding land-masked
neighbourhood-averaged features (5×5 and 9×9 windows) for both `swvl1` and
`log1p(precip_sum24h)`, and switching to a Random Forest to capture
non-linearity, raised held-out R² to **0.74** (MAE ≈ 5.95 on the 0–100
scale). Feature importance confirms the 5×5-neighbourhood 24h rainfall
accumulation dominates (~82%) — local single-cell values matter comparatively
little.

In [ ]:
import numpy as np
from scipy.ndimage import uniform_filter
from sklearn.ensemble import RandomForestRegressor

# --- Reproducible fit: stream a diverse sample of hours, build features,
#     fit + validate the Random Forest out-of-sample. ---

import pandas as pd
from collections import deque

CSV_PATH = r"W:\Dissertation\merged_output_hourly_imerg\merged_data_normalised.csv"
N_LAT, N_LON = 140, 131
CELLS_PER_HOUR = N_LAT * N_LON
N_HOURS_SAMPLE = 600
WINDOW = 24

cols_needed = ["era5_swvl1", "imerg_precipitation", "flood_composite_6h_ahead",
               "is_land", "soil_no_flood_soil", "soil_average_soil",
               "soil_flood_soil", "soil_heavy_flood_soil"]

reader = pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CELLS_PER_HOUR,
                      on_bad_lines="skip")

swvl1_grid  = np.full((N_HOURS_SAMPLE, CELLS_PER_HOUR), np.nan, dtype=np.float32)
precip_grid = np.full((N_HOURS_SAMPLE, CELLS_PER_HOUR), np.nan, dtype=np.float32)
target_grid = np.full((N_HOURS_SAMPLE, CELLS_PER_HOUR), np.nan, dtype=np.float32)
soil_grid = None
land_mask = None

for h, chunk in enumerate(reader):
    if h >= N_HOURS_SAMPLE:
        break
    if len(chunk) != CELLS_PER_HOUR:
        continue
    if land_mask is None:
        land_mask = chunk["is_land"].to_numpy() > 0
        soil_grid = chunk[["soil_no_flood_soil", "soil_average_soil",
                            "soil_flood_soil", "soil_heavy_flood_soil"]].to_numpy()
    swvl1_grid[h]  = chunk["era5_swvl1"].to_numpy()
    precip_grid[h] = chunk["imerg_precipitation"].to_numpy()
    target_grid[h] = chunk["flood_composite_6h_ahead"].to_numpy()

print("Collected", N_HOURS_SAMPLE, "hours of sample data")

In [ ]:
land2d = land_mask.reshape(N_LAT, N_LON)


def spatial_smooth(flat, size):
    grid = flat.reshape(N_LAT, N_LON).astype(np.float64)
    valid = land2d & np.isfinite(grid)
    valid_f = valid.astype(np.float64)
    grid_filled = np.where(valid, grid, 0.0)
    num = uniform_filter(grid_filled, size=size, mode="nearest")
    denom = uniform_filter(valid_f, size=size, mode="nearest")
    sm = np.divide(num, denom, out=np.full_like(num, np.nan), where=denom > 1e-9)
    return sm.reshape(-1)


def feat(h, nbhd_sizes=(5, 9)):
    pw = precip_grid[max(0, h - WINDOW + 1):h + 1].sum(axis=0)
    p24 = np.clip(pw, 0, None)
    log_p24 = np.log1p(p24)
    s = swvl1_grid[h]
    cols = [s, log_p24]
    for size in nbhd_sizes:
        cols.append(spatial_smooth(log_p24, size))
        cols.append(spatial_smooth(s, size))
    cols += [soil_grid[:, 0], soil_grid[:, 1], soil_grid[:, 2], soil_grid[:, 3]]
    return np.column_stack(cols)


def build(h_range):
    Xs, ys = [], []
    for h in h_range:
        base = target_grid[h - 6]
        F = feat(h)
        mask = land_mask & np.isfinite(base) & np.isfinite(F).all(axis=1)
        Xs.append(F[mask])
        ys.append(base[mask])
    return np.concatenate(Xs), np.concatenate(ys)


train_range = range(WINDOW, 450)
val_range = range(450, N_HOURS_SAMPLE - 6)

Xtr, ytr = build(train_range)
Xval, yval = build(val_range)
print("Train rows:", len(Xtr), " Held-out (later, unseen hours) rows:", len(Xval))

rf = RandomForestRegressor(n_estimators=300, max_depth=14, min_samples_leaf=5,
                            n_jobs=-1, random_state=0)
rng = np.random.default_rng(0)
sub_idx = rng.choice(len(Xtr), size=min(500000, len(Xtr)), replace=False)
rf.fit(Xtr[sub_idx], ytr[sub_idx])

pred_val = rf.predict(Xval)
r2_val = 1 - np.sum((yval - pred_val)**2) / np.sum((yval - yval.mean())**2)
print(f"Held-out R^2 = {r2_val:.4f}")
print(f"Held-out MAE = {np.abs(yval - pred_val).mean():.3f}")
print("Feature importances [swvl1, log_p24, log_p24_5, swvl1_5, log_p24_9, "
      "swvl1_9, soil_no, soil_avg, soil_flood, soil_heavy]:")
print(rf.feature_importances_)

import joblib
import os
os.makedirs(SEAGATE + r"\experimental_approx_target", exist_ok=True)
joblib.dump(rf, SEAGATE + r"\experimental_approx_target\rf_model.joblib")
print("Saved rf_model.joblib")

## 4b. Cross-check against a user-recalled formula

The user partially recalled the original approach: `0.6 * precipitation_rate
+/* 0.4 * soil_water_volume_layer1`, normalised to 0-100, with the exact
operator (+ vs *) and normalisation method forgotten.

Testing this directly (best affine-calibrated fit across several
normalisation/window variants):

| Variant | R\u00b2 |
|---|---|
| Instantaneous `imerg_precipitation` + `era5_swvl1`, min-max/percentile, additive or multiplicative | 0.18 (best) |
| Same, with `precipitation_rate` reinterpreted as 24h-accumulated rainfall | **0.40** (best, at 24-30h window) |
| Same, + spatial-neighbourhood smoothing | 0.32-0.33 (no improvement) |

**Conclusion.** The recalled 0.6/0.4 weighting on these two variables is
directionally correct and captures real signal (R\u00b2 jumps from 0.18 to 0.40
once a 24h accumulation window is added, consistent with the window found
independently in section 4) \u2014 a useful partial corroboration of the
reverse-engineering above. But it does not reach the Random Forest's 0.74,
most likely because the true original normalisation was non-linear
(rainfall is heavily zero-inflated / right-skewed, so a raw min-max or
percentile transform under-represents it relative to a log-style transform)
and/or included a spatial catchment term not captured by a simple per-cell
weighted sum. The Random Forest target was kept as the working
reconstruction for `ConvLSTM2`/`CNN3D2` on this basis.

In [ ]:
import numpy as np


def minmax_0_100(arr, lo, hi):
    return np.clip((arr - lo) / (hi - lo) * 100, 0, 100)


def r2_affine(pred, y):
    A = np.column_stack([pred, np.ones(len(pred))])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    fitted = A @ coef
    ss_res = np.sum((y - fitted) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot, coef


# Using the swvl1_grid / precip_grid / target_grid sample collected in
# section 4 above.
WINDOW = 24
rows_base, rows_p, rows_s = [], [], []
for h in range(WINDOW, N_HOURS_SAMPLE):
    base = target_grid[h - 6]
    pw = np.clip(precip_grid[h - WINDOW + 1:h + 1].sum(axis=0), 0, None)
    s = swvl1_grid[h]
    mask = land_mask & np.isfinite(base) & np.isfinite(pw) & np.isfinite(s)
    rows_base.append(base[mask])
    rows_p.append(pw[mask])
    rows_s.append(s[mask])
base_all = np.concatenate(rows_base)
p_all = np.concatenate(rows_p)
s_all = np.concatenate(rows_s)

p_lo, p_hi = np.percentile(p_all, [0.5, 99.5])
s_lo, s_hi = np.percentile(s_all, [0.5, 99.5])
combo = 0.6 * minmax_0_100(p_all, p_lo, p_hi) + 0.4 * minmax_0_100(s_all, s_lo, s_hi)
score, coef = r2_affine(combo, base_all)
print(f"0.6*precip_sum24h + 0.4*swvl1, affine-calibrated R^2 = {score:.4f}")

## 5. Building the approximate target for the full dataset

One full streaming pass over the 77GB CSV, maintaining a rolling 24-hour
precipitation buffer per cell, computing the same spatial-neighbourhood
features at every hour, and predicting the approximate base composite with
the fitted Random Forest. The result is shifted by 6 hours (matching the
original `flood_composite_6h_ahead` convention) and sliced into
train/val/test using the *same* hour boundaries as the existing
`train_X.npy` / `val_X.npy` / `test_X.npy` (15336 / 2208 / 4344 hours), so
the approximate targets align exactly with the existing feature arrays —
no need to rebuild `X`.

Output (all under `experimental_approx_target/`, nothing in the original
locations is touched):
- `base_composite_approx_full.npy` — (21888, 140, 131)
- `train_y_approx.npy`, `val_y_approx.npy`, `test_y_approx.npy`

This step takes roughly 30-90 minutes (single sequential pass over the full
CSV) and was run standalone as `build_approx_target.py`; the full source is
reproduced below for reference.

In [ ]:
import os
import sys
import time
from collections import deque

import numpy as np
import pandas as pd
import joblib
from scipy.ndimage import uniform_filter

CSV_PATH = r"W:\Dissertation\merged_output_hourly_imerg\merged_data_normalised.csv"
SEAGATE = r"W:\Dissertation\Volumes\Seagate_2TB"
OUT_DIR = os.path.join(SEAGATE, "experimental_approx_target")
os.makedirs(OUT_DIR, exist_ok=True)

N_LAT, N_LON = 140, 131
CELLS_PER_HOUR = N_LAT * N_LON
N_HOURS_TOTAL = 21888  # 15336 train + 2208 val + 4344 test
WINDOW = 24

RF_PATH = r"C:\Users\zd827032\AppData\Local\Temp\claude\w--Dissertation\f2313256-a4c8-4898-942f-c35de8216148\scratchpad\rf_model.joblib"
rf = joblib.load(RF_PATH)
print("Loaded RF model", flush=True)

cols_needed = ["era5_swvl1", "imerg_precipitation", "is_land",
               "soil_no_flood_soil", "soil_average_soil", "soil_flood_soil", "soil_heavy_flood_soil"]

reader = pd.read_csv(CSV_PATH, usecols=cols_needed, chunksize=CELLS_PER_HOUR,
                      on_bad_lines="skip")

land2d = None
land_flat = None
soil_flat = None
precip_buffer = deque(maxlen=WINDOW)

base_composite = np.full((N_HOURS_TOTAL, N_LAT, N_LON), np.nan, dtype=np.float32)

t0 = time.time()

def spatial_smooth(flat, size, land2d_local):
    grid = flat.reshape(N_LAT, N_LON).astype(np.float64)
    valid = land2d_local & np.isfinite(grid)
    valid_f = valid.astype(np.float64)
    grid_filled = np.where(valid, grid, 0.0)
    num = uniform_filter(grid_filled, size=size, mode="nearest")
    denom = uniform_filter(valid_f, size=size, mode="nearest")
    sm = np.divide(num, denom, out=np.full_like(num, np.nan), where=denom > 1e-9)
    return sm.reshape(-1)


for h, chunk in enumerate(reader):
    if h >= N_HOURS_TOTAL:
        break
    if len(chunk) != CELLS_PER_HOUR:
        print(f"WARNING: hour {h} malformed chunk (len={len(chunk)}), skipping", flush=True)
        continue

    if land_flat is None:
        land_flat = chunk["is_land"].to_numpy() > 0
        land2d = land_flat.reshape(N_LAT, N_LON)
        soil_flat = chunk[["soil_no_flood_soil", "soil_average_soil",
                            "soil_flood_soil", "soil_heavy_flood_soil"]].to_numpy()

    swvl1_h = chunk["era5_swvl1"].to_numpy()
    precip_h = chunk["imerg_precipitation"].to_numpy()
    precip_buffer.append(precip_h)

    p_window = np.sum(np.array(precip_buffer), axis=0)
    p24 = np.clip(p_window, 0, None)
    log_p24 = np.log1p(p24)

    log_p24_5 = spatial_smooth(log_p24, 5, land2d)
    swvl1_5 = spatial_smooth(swvl1_h, 5, land2d)
    log_p24_9 = spatial_smooth(log_p24, 9, land2d)
    swvl1_9 = spatial_smooth(swvl1_h, 9, land2d)

    F = np.column_stack([swvl1_h, log_p24, log_p24_5, swvl1_5, log_p24_9, swvl1_9,
                          soil_flat[:, 0], soil_flat[:, 1], soil_flat[:, 2], soil_flat[:, 3]])

    valid_mask = land_flat & np.isfinite(F).all(axis=1)
    if valid_mask.sum() > 0:
        pred = rf.predict(F[valid_mask])
        grid = np.full(CELLS_PER_HOUR, np.nan, dtype=np.float32)
        grid[valid_mask] = pred
        base_composite[h] = grid.reshape(N_LAT, N_LON)

    if h % 500 == 0 or h == N_HOURS_TOTAL - 1:
        elapsed = time.time() - t0
        rate = (h + 1) / elapsed if elapsed > 0 else 0
        eta_min = (N_HOURS_TOTAL - h - 1) / rate / 60 if rate > 0 else float("nan")
        print(f"hour {h:6d}/{N_HOURS_TOTAL}  elapsed={elapsed/60:.1f}min  "
              f"rate={rate:.2f}h/s  ETA={eta_min:.1f}min", flush=True)

print(f"\nDone streaming. Total time: {(time.time()-t0)/60:.1f} min", flush=True)
np.save(os.path.join(OUT_DIR, "base_composite_approx_full.npy"), base_composite)
print("Saved base_composite_approx_full.npy", base_composite.shape, flush=True)

# Shift 6 hours ahead: target_approx(h) = base_composite(h+6)
target_approx = np.full((N_HOURS_TOTAL, N_LAT, N_LON), np.nan, dtype=np.float32)
target_approx[:N_HOURS_TOTAL - 6] = base_composite[6:]

TRAIN_END = 15336
VAL_END = TRAIN_END + 2208

train_y_approx = target_approx[:TRAIN_END]
val_y_approx = target_approx[TRAIN_END:VAL_END]
test_y_approx = target_approx[VAL_END:]

np.save(os.path.join(OUT_DIR, "train_y_approx.npy"), train_y_approx)
np.save(os.path.join(OUT_DIR, "val_y_approx.npy"), val_y_approx)
np.save(os.path.join(OUT_DIR, "test_y_approx.npy"), test_y_approx)
print("Saved train/val/test_y_approx.npy:", train_y_approx.shape, val_y_approx.shape, test_y_approx.shape, flush=True)
print("ALL DONE", flush=True)


## 6. Training the experimental models

`ConvLSTM_approx` and `CNN3D_approx` are trained with **identical
architecture and hyperparameters** to the original `ConvLSTM`/`CNN3D` runs
in `ML_MODELS.ipynb` — only the target array differs (`train_y_approx.npy` /
`val_y_approx.npy` instead of `train_y.npy` / `val_y.npy`). Checkpoints save
to `model_checkpoints/ConvLSTM_approx/` and `model_checkpoints/CNN3D_approx/`,
logs to `convlstm_log_approx.csv` / `cnn3d_log_approx.csv` — the original
checkpoints and logs are never overwritten. Run standalone as
`train_convlstm_approx.py` / `train_cnn3d_approx.py` (full source below).

In [ ]:
# TRAIN CONVLSTM ON THE APPROXIMATE (RECONSTRUCTED) TARGET — EXPERIMENTAL
# ============================================================================
# Identical architecture/hyperparameters to the original ConvLSTM training
# run in ML_MODELS.ipynb. The only difference: train_y/val_y point at
# train_y_approx.npy / val_y_approx.npy (built by build_approx_target.py
# from a Random Forest reconstruction of flood_composite_6h_ahead, since
# the original generating code for that column was lost). Checkpoints and
# logs go to separate "_approx" locations so the original results are
# never touched.
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from convlstm_model import ConvLSTMForecaster, WeightedPinballLoss

# ── Paths ──
SEAGATE        = r"W:\Dissertation\Volumes\Seagate_2TB"
APPROX_DIR     = os.path.join(SEAGATE, "experimental_approx_target")
TRAIN_X_PATH   = os.path.join(SEAGATE, "train_X.npy")
TRAIN_Y_PATH   = os.path.join(APPROX_DIR, "train_y_approx.npy")
VAL_X_PATH     = os.path.join(SEAGATE, "val_X.npy")
VAL_Y_PATH     = os.path.join(APPROX_DIR, "val_y_approx.npy")
CHECKPOINT_DIR = os.path.join(SEAGATE, "model_checkpoints", "ConvLSTM_approx")
LOG_DIR        = os.path.join(SEAGATE, "model_logs")
CSV_PATH       = r"W:\Dissertation\merged_output_hourly_imerg\merged_data_normalised.csv"

# ── Config (identical to original ConvLSTM run) ──
N_LAT       = 140
N_LON       = 131
N_FEATURES  = 20
N_QUANTILES = 10
SEQ_LEN     = 12
PATCH_SIZE  = 64
HALF        = PATCH_SIZE // 2
BATCH_SIZE  = 48
N_EPOCHS    = 40
LR          = 3e-4
WARMUP_EPOCHS = 3
DROPOUT     = 0.15
PATIENCE    = 10
SAVE_EVERY  = 5
TARGET_COL  = "flood_composite_6h_ahead"


class BinaryPatchDataset(Dataset):
    def __init__(self, X, y, centres, n_hours):
        self.X        = X
        self.y        = y
        self.centres  = centres
        self.seq_ends = list(range(SEQ_LEN, n_hours))

    def __len__(self):
        return len(self.seq_ends)

    def _patch(self, grid, lat_c, lon_c):
        return grid[...,
                    lat_c - HALF:lat_c + HALF,
                    lon_c - HALF:lon_c + HALF].copy()

    def __getitem__(self, idx):
        seq_end      = self.seq_ends[idx]
        ci           = np.random.randint(0, len(self.centres))
        lat_c, lon_c = self.centres[ci]
        x_seq = np.stack([
            self._patch(self.X[h], lat_c, lon_c)
            for h in range(seq_end - SEQ_LEN, seq_end)
        ])
        y = self._patch(self.y[seq_end][np.newaxis], lat_c, lon_c)[0]
        return (torch.from_numpy(x_seq).float(),
                torch.from_numpy(y).float())


if __name__ == '__main__':
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark        = True
        torch.backends.cuda.matmul.allow_tf32 = True
        print(f"GPU:  {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    else:
        print("CUDA not available - running on CPU")

    print("\nLoading binary data into RAM...")
    t0 = time.time()

    print("  Loading train_X + train_y_approx...")
    train_X = np.load(TRAIN_X_PATH)
    train_y = np.load(TRAIN_Y_PATH)
    print(f"  Train: {train_X.shape} - {train_X.nbytes/1e9:.1f} GB")

    print("  Loading val_X + val_y_approx...")
    val_X = np.load(VAL_X_PATH)
    val_y = np.load(VAL_Y_PATH)
    print(f"  Val:   {val_X.shape} - {val_X.nbytes/1e9:.1f} GB")
    print(f"  Total load time: {(time.time()-t0)/60:.1f} min")

    N_TRAIN = train_X.shape[0]
    N_VAL   = val_X.shape[0]

    print("\nBuilding land cell index...")
    header_cols  = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
    SKIP_COLS    = {"time", "latitude", "longitude", TARGET_COL}
    FEATURE_COLS = [c for c in header_cols if c not in SKIP_COLS]
    land_idx     = FEATURE_COLS.index("is_land")

    land_mask = train_X[0, land_idx]
    if land_mask.max() == 0:
        print("  is_land channel all zero - using soil_average_soil instead")
        soil_idx  = FEATURE_COLS.index("soil_average_soil")
        land_mask = train_X[0, soil_idx]

    valid_centres = [
        (lat, lon)
        for lat in range(HALF, N_LAT - HALF)
        for lon in range(HALF, N_LON - HALF)
        if land_mask[lat, lon] > 0.0
    ]
    print(f"Valid patch centres: {len(valid_centres):,}")

    print("\nStandardising features (train-only stats)...")
    chan_mean = train_X.mean(axis=(0, 2, 3), keepdims=True).astype(np.float32)
    chan_std  = train_X.std(axis=(0, 2, 3), keepdims=True).astype(np.float32)
    chan_std[chan_std < 1e-6] = 1.0
    np.save(os.path.join(CHECKPOINT_DIR, "feature_mean.npy"), chan_mean)
    np.save(os.path.join(CHECKPOINT_DIR, "feature_std.npy"), chan_std)
    train_X -= chan_mean
    train_X /= chan_std
    val_X   -= chan_mean
    val_X   /= chan_std
    print(f"  Done. Stats saved to {CHECKPOINT_DIR}")

    print("\nBuilding datasets...")
    train_dataset = BinaryPatchDataset(train_X, train_y, valid_centres, N_TRAIN)
    val_dataset   = BinaryPatchDataset(val_X, val_y, valid_centres, N_VAL)
    print(f"  Train sequences: {len(train_dataset):,}")
    print(f"  Val sequences:   {len(val_dataset):,}")

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE,
        shuffle=True,  num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=True)

    val_loader = DataLoader(
        val_dataset,   batch_size=BATCH_SIZE,
        shuffle=False, num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=False)

    print(f"  Train batches/epoch: {len(train_loader):,}")
    print(f"  Val batches/epoch:   {len(val_loader):,}")

    model = ConvLSTMForecaster(
        in_channels=N_FEATURES,
        hidden_dims=(64, 128, 64),
        n_quantiles=N_QUANTILES,
        dropout=DROPOUT).to(device)

    if torch.cuda.device_count() > 1:
        print(f"\nUsing {torch.cuda.device_count()} GPUs with DataParallel")
        model = torch.nn.DataParallel(model)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"\nConvLSTM parameters: {n_params:,}")

    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR * 0.1, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)
    criterion = WeightedPinballLoss().to(device)
    scaler = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

    best_val   = np.inf
    patience_c = 0
    log        = []
    t_start    = time.time()

    print(f"\nStarting ConvLSTM (approx target) - {N_EPOCHS} epochs")
    print(f"Batch: {BATCH_SIZE} | Patch: {PATCH_SIZE}x{PATCH_SIZE}")
    print(f"Checkpoints -> {CHECKPOINT_DIR}\n")

    with tqdm(total=N_EPOCHS, desc="ConvLSTM-approx", unit="epoch",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} epochs "
                         "[{elapsed}<{remaining}] {postfix}") as ebar:

        for epoch in range(N_EPOCHS):
            t_ep = time.time()

            if epoch < WARMUP_EPOCHS:
                warmup_lr = LR * 0.1 + (LR - LR * 0.1) * (epoch + 1) / WARMUP_EPOCHS
                for g in optimizer.param_groups:
                    g['lr'] = warmup_lr

            model.train()
            train_loss = 0.0
            n_train    = 0

            for x, y in train_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                    pred = model(x)
                    loss = criterion(pred, y)
                if torch.isnan(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
                n_train    += 1

            avg_train = train_loss / max(n_train, 1)

            model.eval()
            val_loss = 0.0
            n_val    = 0

            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                        pred = model(x, sort_quantiles=True)
                        loss = criterion(pred, y)
                    if not torch.isnan(loss):
                        val_loss += loss.item()
                        n_val    += 1

            avg_val   = val_loss / max(n_val, 1)
            epoch_min = (time.time() - t_ep) / 60
            if epoch >= WARMUP_EPOCHS:
                scheduler.step(avg_val)
            cur_lr = optimizer.param_groups[0]['lr']

            vram_gb = (torch.cuda.memory_allocated() / 1e9
                       if device.type == "cuda" else 0.0)

            log.append({"epoch": epoch+1, "train_loss": avg_train,
                        "val_loss": avg_val, "lr": cur_lr,
                        "epoch_min": epoch_min})

            ebar.set_postfix(
                train=f"{avg_train:.4f}", val=f"{avg_val:.4f}",
                best=f"{best_val:.4f}", lr=f"{cur_lr:.2e}",
                ep_min=f"{epoch_min:.1f}min",
                vram=f"{vram_gb:.1f}GB")
            ebar.update(1)
            print(f"epoch {epoch+1}/{N_EPOCHS}  train={avg_train:.4f}  "
                  f"val={avg_val:.4f}  best={best_val:.4f}  "
                  f"{epoch_min:.1f}min/epoch", flush=True)

            if avg_val < best_val:
                best_val   = avg_val
                patience_c = 0
                model_state = (model.module.state_dict()
                               if isinstance(model, torch.nn.DataParallel)
                               else model.state_dict())
                torch.save({
                    "epoch":           epoch,
                    "model_state":     model_state,
                    "optimizer_state": optimizer.state_dict(),
                    "val_loss":        avg_val,
                }, os.path.join(CHECKPOINT_DIR, "best.pt"))
                print(f"  best val: {avg_val:.4f} -> best.pt", flush=True)
            else:
                patience_c += 1
                if patience_c >= PATIENCE:
                    print(f"Early stop at epoch {epoch+1}", flush=True)
                    break

            if (epoch+1) % SAVE_EVERY == 0:
                model_state = (model.module.state_dict()
                               if isinstance(model, torch.nn.DataParallel)
                               else model.state_dict())
                torch.save({
                    "epoch":       epoch,
                    "model_state": model_state,
                    "val_loss":    avg_val,
                }, os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1:04d}.pt"))

    pd.DataFrame(log).to_csv(
        os.path.join(LOG_DIR, "convlstm_log_approx.csv"), index=False)
    elapsed_total = (time.time() - t_start) / 60
    print(f"\nConvLSTM (approx) complete in {elapsed_total:.0f} min", flush=True)
    print(f"  Best val loss: {best_val:.4f}", flush=True)
    print(f"  Log:           {LOG_DIR}\\convlstm_log_approx.csv", flush=True)
    print(f"  Checkpoint:    {CHECKPOINT_DIR}\\best.pt", flush=True)


In [ ]:
# TRAIN 3D-CNN ON THE APPROXIMATE (RECONSTRUCTED) TARGET — EXPERIMENTAL
# ============================================================================
# Identical architecture/hyperparameters to the original 3D-CNN training run
# in ML_MODELS.ipynb. The only difference: train_y/val_y point at
# train_y_approx.npy / val_y_approx.npy. Checkpoints and logs go to separate
# "_approx" locations so the original results are never touched.
# ============================================================================
import os
import time
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from convlstm_model import WeightedPinballLoss
from cnn3d_model import CNN3DForecaster

# ── Paths ──
SEAGATE        = r"W:\Dissertation\Volumes\Seagate_2TB"
APPROX_DIR     = os.path.join(SEAGATE, "experimental_approx_target")
TRAIN_X_PATH   = os.path.join(SEAGATE, "train_X.npy")
TRAIN_Y_PATH   = os.path.join(APPROX_DIR, "train_y_approx.npy")
VAL_X_PATH     = os.path.join(SEAGATE, "val_X.npy")
VAL_Y_PATH     = os.path.join(APPROX_DIR, "val_y_approx.npy")
CHECKPOINT_DIR = os.path.join(SEAGATE, "model_checkpoints", "CNN3D_approx")
LOG_DIR        = os.path.join(SEAGATE, "model_logs")
CSV_PATH       = r"W:\Dissertation\merged_output_hourly_imerg\merged_data_normalised.csv"

# ── Config (identical to original CNN3D run) ──
N_LAT       = 140
N_LON       = 131
N_FEATURES  = 20
N_QUANTILES = 10
SEQ_LEN     = 12
PATCH_SIZE  = 64
HALF        = PATCH_SIZE // 2
BATCH_SIZE  = 96
N_EPOCHS    = 40
LR          = 5e-4
WARMUP_EPOCHS = 3
DROPOUT     = 0.15
PATIENCE    = 10
SAVE_EVERY  = 5
TARGET_COL  = "flood_composite_6h_ahead"


class BinaryPatchDataset(Dataset):
    def __init__(self, X, y, centres, n_hours):
        self.X        = X
        self.y        = y
        self.centres  = centres
        self.seq_ends = list(range(SEQ_LEN, n_hours))

    def __len__(self):
        return len(self.seq_ends)

    def _patch(self, grid, lat_c, lon_c):
        return grid[...,
                    lat_c - HALF:lat_c + HALF,
                    lon_c - HALF:lon_c + HALF].copy()

    def __getitem__(self, idx):
        seq_end      = self.seq_ends[idx]
        ci           = np.random.randint(0, len(self.centres))
        lat_c, lon_c = self.centres[ci]
        x_seq = np.stack([
            self._patch(self.X[h], lat_c, lon_c)
            for h in range(seq_end - SEQ_LEN, seq_end)
        ])
        y = self._patch(self.y[seq_end][np.newaxis], lat_c, lon_c)[0]
        return (torch.from_numpy(x_seq).float(),
                torch.from_numpy(y).float())


if __name__ == '__main__':
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(LOG_DIR, exist_ok=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark        = True
        torch.backends.cuda.matmul.allow_tf32 = True
        print(f"GPU:  {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    else:
        print("CUDA not available - running on CPU")

    print("\nLoading binary data into RAM...")
    t0 = time.time()

    print("  Loading train_X + train_y_approx...")
    train_X = np.load(TRAIN_X_PATH)
    train_y = np.load(TRAIN_Y_PATH)
    print(f"  Train: {train_X.shape} - {train_X.nbytes/1e9:.1f} GB")

    print("  Loading val_X + val_y_approx...")
    val_X = np.load(VAL_X_PATH)
    val_y = np.load(VAL_Y_PATH)
    print(f"  Val:   {val_X.shape} - {val_X.nbytes/1e9:.1f} GB")
    print(f"  Total load time: {(time.time()-t0)/60:.1f} min")

    N_TRAIN = train_X.shape[0]
    N_VAL   = val_X.shape[0]

    print("\nBuilding land cell index...")
    header_cols  = pd.read_csv(CSV_PATH, nrows=0).columns.tolist()
    SKIP_COLS    = {"time", "latitude", "longitude", TARGET_COL}
    FEATURE_COLS = [c for c in header_cols if c not in SKIP_COLS]
    land_idx     = FEATURE_COLS.index("is_land")

    land_mask = train_X[0, land_idx]
    if land_mask.max() == 0:
        print("  is_land channel all zero - using soil_average_soil instead")
        soil_idx  = FEATURE_COLS.index("soil_average_soil")
        land_mask = train_X[0, soil_idx]

    valid_centres = [
        (lat, lon)
        for lat in range(HALF, N_LAT - HALF)
        for lon in range(HALF, N_LON - HALF)
        if land_mask[lat, lon] > 0.0
    ]
    print(f"Valid patch centres: {len(valid_centres):,}")

    print("\nStandardising features (train-only stats)...")
    chan_mean = train_X.mean(axis=(0, 2, 3), keepdims=True).astype(np.float32)
    chan_std  = train_X.std(axis=(0, 2, 3), keepdims=True).astype(np.float32)
    chan_std[chan_std < 1e-6] = 1.0
    np.save(os.path.join(CHECKPOINT_DIR, "feature_mean.npy"), chan_mean)
    np.save(os.path.join(CHECKPOINT_DIR, "feature_std.npy"), chan_std)
    train_X -= chan_mean
    train_X /= chan_std
    val_X   -= chan_mean
    val_X   /= chan_std
    print(f"  Done. Stats saved to {CHECKPOINT_DIR}")

    print("\nBuilding datasets...")
    train_dataset = BinaryPatchDataset(train_X, train_y, valid_centres, N_TRAIN)
    val_dataset   = BinaryPatchDataset(val_X, val_y, valid_centres, N_VAL)
    print(f"  Train sequences: {len(train_dataset):,}")
    print(f"  Val sequences:   {len(val_dataset):,}")

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE,
        shuffle=True,  num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=True)

    val_loader = DataLoader(
        val_dataset,   batch_size=BATCH_SIZE,
        shuffle=False, num_workers=0,
        pin_memory=(device.type == "cuda"),
        drop_last=False)

    print(f"  Train batches/epoch: {len(train_loader):,}")
    print(f"  Val batches/epoch:   {len(val_loader):,}")

    model     = CNN3DForecaster(
        in_channels=N_FEATURES,
        n_quantiles=N_QUANTILES,
        dropout=DROPOUT).to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR * 0.1, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6)
    criterion = WeightedPinballLoss().to(device)
    scaler    = torch.amp.GradScaler('cuda', enabled=(device.type == "cuda"))

    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n3D-CNN parameters: {n_params:,}")

    best_val   = np.inf
    patience_c = 0
    log        = []
    t_start    = time.time()

    print(f"\nStarting 3D-CNN (approx target) - {N_EPOCHS} epochs")
    print(f"Batch: {BATCH_SIZE} | Patch: {PATCH_SIZE}x{PATCH_SIZE}")
    print(f"Checkpoints -> {CHECKPOINT_DIR}\n")

    with tqdm(total=N_EPOCHS, desc="CNN3D-approx", unit="epoch",
              bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} epochs "
                         "[{elapsed}<{remaining}] {postfix}") as ebar:

        for epoch in range(N_EPOCHS):
            t_ep = time.time()

            if epoch < WARMUP_EPOCHS:
                warmup_lr = LR * 0.1 + (LR - LR * 0.1) * (epoch + 1) / WARMUP_EPOCHS
                for g in optimizer.param_groups:
                    g['lr'] = warmup_lr

            model.train()
            train_loss = 0.0
            n_train    = 0

            for x, y in train_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                    pred = model(x)
                    loss = criterion(pred, y)
                if torch.isnan(loss):
                    continue
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                train_loss += loss.item()
                n_train    += 1

            avg_train = train_loss / max(n_train, 1)

            model.eval()
            val_loss = 0.0
            n_val    = 0

            with torch.no_grad():
                for x, y in val_loader:
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)
                    with torch.amp.autocast('cuda', enabled=(device.type == "cuda")):
                        pred = model(x, sort_quantiles=True)
                        loss = criterion(pred, y)
                    if not torch.isnan(loss):
                        val_loss += loss.item()
                        n_val    += 1

            avg_val   = val_loss / max(n_val, 1)
            epoch_min = (time.time() - t_ep) / 60
            if epoch >= WARMUP_EPOCHS:
                scheduler.step(avg_val)
            cur_lr = optimizer.param_groups[0]['lr']

            vram_gb = (torch.cuda.memory_allocated() / 1e9
                       if device.type == "cuda" else 0.0)

            log.append({"epoch": epoch+1, "train_loss": avg_train,
                        "val_loss": avg_val, "lr": cur_lr,
                        "epoch_min": epoch_min})

            ebar.set_postfix(
                train=f"{avg_train:.4f}", val=f"{avg_val:.4f}",
                best=f"{best_val:.4f}", lr=f"{cur_lr:.2e}",
                ep_min=f"{epoch_min:.1f}min",
                vram=f"{vram_gb:.1f}GB")
            ebar.update(1)
            print(f"epoch {epoch+1}/{N_EPOCHS}  train={avg_train:.4f}  "
                  f"val={avg_val:.4f}  best={best_val:.4f}  "
                  f"{epoch_min:.1f}min/epoch", flush=True)

            if avg_val < best_val:
                best_val   = avg_val
                patience_c = 0
                torch.save({
                    "epoch":           epoch,
                    "model_state":     model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "val_loss":        avg_val,
                }, os.path.join(CHECKPOINT_DIR, "best.pt"))
                print(f"  best val: {avg_val:.4f} -> best.pt", flush=True)
            else:
                patience_c += 1
                if patience_c >= PATIENCE:
                    print(f"Early stop at epoch {epoch+1}", flush=True)
                    break

            if (epoch+1) % SAVE_EVERY == 0:
                torch.save({
                    "epoch":       epoch,
                    "model_state": model.state_dict(),
                    "val_loss":    avg_val,
                }, os.path.join(CHECKPOINT_DIR, f"epoch_{epoch+1:04d}.pt"))

    pd.DataFrame(log).to_csv(
        os.path.join(LOG_DIR, "cnn3d_log_approx.csv"), index=False)
    elapsed_total = (time.time() - t_start) / 60
    print(f"\n3D-CNN (approx) complete in {elapsed_total:.0f} min", flush=True)
    print(f"  Best val loss: {best_val:.4f}", flush=True)
    print(f"  Log:           {LOG_DIR}\\cnn3d_log_approx.csv", flush=True)
    print(f"  Checkpoint:    {CHECKPOINT_DIR}\\best.pt", flush=True)


## 7. Limitations

- **This is a reconstruction, not a recovery.** The exact original formula
  and normalisation method for `flood_composite` were lost and could not be
  found anywhere in this repository. What is implemented here is the
  best-validated approximation found by searching feature/window/model
  space against the real recorded target values, using a Random Forest
  rather than a closed-form equation.
- **Held-out R² = 0.74, not 1.0.** ~26% of the variance in the real target
  is not explained by this reconstruction. Differences between the original
  and `_approx` model results should be interpreted with that in mind —
  they reflect both genuine model behaviour *and* target-reconstruction
  error, which cannot be cleanly separated.
- **The original `ML_MODELS.ipynb` results remain the primary, valid
  results** for this dissertation. This experimental branch exists
  alongside them, not in place of them.
- **~12.8% speckled gaps in the reconstructed target maps.**
  `build_approx_target.py` streams `imerg_precipitation` directly from the
  raw CSV to compute the 24h rolling rainfall sum, but does not zero-fill
  IMERG's satellite swath gaps the way the original `train_X`/`val_X`/
  `test_X` conversion does. Wherever a cell hits a swath gap, `NaN`
  propagates into the rolling sum and the Random Forest skips that cell,
  leaving a hole in `base_composite_approx_full.npy` (visible as white
  speckle in `test_spatial_maps2.png`'s "Actual" panel). Confirmed via
  direct inspection: at test hour 2107, all 645 speckled cells (12.8% of
  land) correspond exactly to cells that are zero-filled swath gaps in
  `test_X.npy`. This does not corrupt model training \u2014
  `WeightedPinballLoss` already masks `NaN` targets out of the loss \u2014 it
  only means ConvLSTM2/CNN3D2 saw ~13% less supervision per hour than they
  could have. Left unfixed by user decision (2026-08-14): not worth another
  ~2.5 hour rebuild+retrain cycle for a coverage improvement rather than a
  correctness fix.